# PPO Self-Play: EdgeTransformerNet Actor + ValueNet Critic

Fine-tunes the imitation-pretrained `EdgeTransformerNet` via PPO self-play.

## Required Kaggle inputs

Add these datasets before running:

| Dataset name (slug) | Contents |
|---|---|
| `orbitswar-src` | Zip of your full project (main.py, imitation/, rl/, classifier/, enemy agents/) |
| `orbitswar-imitation` | `best_model.pt` from `imitation/checkpoints/` after imitation training |
| `orbitswar-rl` *(optional)* | `last.pt` from `rl/` — warm-starts the critic from OrbitsNet weights |

## Output
- `/kaggle/working/rl/transformer_best.pt` — best win-rate checkpoint  
- `/kaggle/working/rl/transformer_last.pt` — latest checkpoint  

Download both from the **Output** tab after the session ends.

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Install orbit_wars environment
# ─────────────────────────────────────────────────────────
!pip install kaggle_environments -q

# Verify orbit_wars is available
from kaggle_environments import make
env = make('orbit_wars', configuration={'episodeSteps': 5})
print('orbit_wars OK')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2b — Logging setup
# Tees ALL print() output to a .log file on disk so you
# can review the full PPO training run after the session.
# Download ppo_training.log from the Output tab.
# ─────────────────────────────────────────────────────────
import sys, time
from pathlib import Path

LOG_PATH = Path('/kaggle/working/ppo_training.log')

class _TeeLogger:
    """Duplicate every print() to a persistent log file."""
    def __init__(self, path):
        self._orig = sys.stdout
        self._file = open(path, 'w', buffering=1, encoding='utf-8')
        ts = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
        self._file.write('# OrbitsWar — PPO Training Log\n')
        self._file.write(f'# Started : {ts}\n\n')
        self._file.flush()
    def write(self, msg):
        self._orig.write(msg)
        try:   self._file.write(msg)
        except: pass
    def flush(self):
        self._orig.flush()
        try:   self._file.flush()
        except: pass
    def isatty(self):
        return False   # keeps tqdm / progress bars happy
    def close(self):
        sys.stdout = self._orig
        try:   self._file.flush(); self._file.close()
        except: pass

_tee = _TeeLogger(LOG_PATH)
sys.stdout = _tee
print(f'Logging to : {LOG_PATH}')
print('(All print() output is mirrored here — download from Output tab.)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Check GPU
# ─────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Dataset paths (edit if your slug names differ)
# ─────────────────────────────────────────────────────────
from pathlib import Path

# ── Input dataset slugs ───────────────────────────────────
SRC_DATASET       = 'orbitswar-src'        # your project zip
IMITATION_DATASET = 'orbitswar-imitation'  # best_model.pt
RL_DATASET        = 'orbitswar-rl'         # last.pt  (optional)

SRC_DIR  = Path(f'/kaggle/input/{SRC_DATASET}')
IMIT_DIR = Path(f'/kaggle/input/{IMITATION_DATASET}')
RL_DIR   = Path(f'/kaggle/input/{RL_DATASET}')

print(f'SRC  exists: {SRC_DIR.exists()}   ({SRC_DIR})')
print(f'IMIT exists: {IMIT_DIR.exists()}  ({IMIT_DIR})')
print(f'RL   exists: {RL_DIR.exists()}   ({RL_DIR})')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Copy source tree to /kaggle/working/
#           (keeps relative imports intact)
# ─────────────────────────────────────────────────────────
import os, sys, shutil, glob

WORK = Path('/kaggle/working')

# Copy everything from the source dataset
for item in SRC_DIR.iterdir():
    dst = WORK / item.name
    if item.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(item, dst)
    else:
        shutil.copy2(item, dst)
print('Source tree copied.')

# ── Copy imitation checkpoint ─────────────────────────────
imit_ckpt_dst = WORK / 'imitation' / 'checkpoints'
imit_ckpt_dst.mkdir(parents=True, exist_ok=True)

imit_src = IMIT_DIR / 'best_model.pt'
if imit_src.exists():
    shutil.copy2(imit_src, imit_ckpt_dst / 'best_model.pt')
    print(f'Imitation checkpoint copied -> {imit_ckpt_dst}/best_model.pt')
else:
    print('WARNING: best_model.pt not found — actor will start from random init.')

# ── Copy RL critic checkpoint (optional) ─────────────────
rl_ckpt_dst = WORK / 'rl'
rl_ckpt_dst.mkdir(parents=True, exist_ok=True)

rl_src = RL_DIR / 'last.pt'
if rl_src.exists():
    shutil.copy2(rl_src, rl_ckpt_dst / 'last.pt')
    print(f'RL checkpoint copied -> {rl_ckpt_dst}/last.pt')
else:
    print('No RL checkpoint — critic starts fresh (random init).')

# ── Ensure enemy agents directory exists ──────────────────
enemy_dir = WORK / 'enemy agents'
if not enemy_dir.exists():
    enemy_dir.mkdir(parents=True)
    print('Created empty enemy agents dir.')

# Add main.py as a self-play bot if no enemies found
std_bots = list(enemy_dir.glob('*/main.py'))
my_bots  = list((enemy_dir / 'my').glob('*.py')) if (enemy_dir / 'my').exists() else []
if not std_bots and not my_bots:
    self_bot_dir = enemy_dir / 'self_main'
    self_bot_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORK / 'main.py', self_bot_dir / 'main.py')
    print('No enemies found — added main.py as self-play opponent.')
else:
    labels = [p.parent.name for p in std_bots] + [p.stem for p in my_bots]
    print(f'Enemies: {labels}')

# ── Python path ───────────────────────────────────────────
for p in [
    str(WORK),
    str(WORK / 'rl'),
    str(WORK / 'imitation'),
    str(WORK / 'classifier'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('\nWorking directory contents:')
for p in sorted(WORK.iterdir()):
    print(f'  {p.name}/'  if p.is_dir() else f'  {p.name}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Verify imports
# ─────────────────────────────────────────────────────────
os.chdir(WORK)   # make relative imports work from project root

from imitation.model    import EdgeTransformerNet, load_model as load_actor
from imitation.features import build_all_candidate_edges

# Count actor parameters
actor_test = EdgeTransformerNet()
n_params = sum(p.numel() for p in actor_test.parameters())
print(f'EdgeTransformerNet params: {n_params:,}')
del actor_test

# Check imitation checkpoint
imit_ckpt = WORK / 'imitation' / 'checkpoints' / 'best_model.pt'
print(f'Imitation checkpoint exists: {imit_ckpt.exists()}')
if imit_ckpt.exists():
    import torch
    ckpt = torch.load(imit_ckpt, map_location='cpu', weights_only=False)
    print(f'  Trained epoch: {ckpt.get("epoch", "?")}  '
          f'val_loss: {ckpt.get("val_loss", float("nan")):.4f}')

print('\nAll imports OK.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Training hyperparameters
#           (tuned for Kaggle T4 GPU, ~9h session)
# ─────────────────────────────────────────────────────────

# Each round = GAMES_PER_ROUND games + 1 PPO update
# T4 estimate: ~3–4 min/round  →  9h session ≈ 130–180 rounds
N_ROUNDS        = 200    # increase if your session allows
GAMES_PER_ROUND = 8      # games collected before each PPO update

# Actor LR is LOW — we are fine-tuning a pretrained imitation model
LR_ACTOR  = 5e-5
LR_CRITIC = 3e-4

ENT_COEF  = 0.01   # low — let policy commit; 1.70 entropy means stuck at random
CLIP_EPS  = 0.2
PPO_EPOCHS = 4

print(f'Rounds         : {N_ROUNDS}')
print(f'Games / round  : {GAMES_PER_ROUND}')
print(f'Total games    : {N_ROUNDS * GAMES_PER_ROUND}')
print(f'LR actor       : {LR_ACTOR}')
print(f'LR critic      : {LR_CRITIC}')
print(f'Entropy coef   : {ENT_COEF}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Patch trainer hyperparameters, then launch
# ─────────────────────────────────────────────────────────
import rl.transformer_trainer as TT

# Override module-level constants before calling train()
TT.N_ROUNDS        = N_ROUNDS
TT.GAMES_PER_ROUND = GAMES_PER_ROUND
TT.LR_ACTOR        = LR_ACTOR
TT.LR_CRITIC       = LR_CRITIC
TT.ENT_COEF        = ENT_COEF
TT.CLIP_EPS        = CLIP_EPS
TT.PPO_EPOCHS      = PPO_EPOCHS

# Override save paths → /kaggle/working/rl/
TT.SAVE_PATH = WORK / 'rl' / 'transformer_last.pt'
TT.BEST_PATH = WORK / 'rl' / 'transformer_best.pt'

print(f'Checkpoints will be saved to:')
print(f'  Last : {TT.SAVE_PATH}')
print(f'  Best : {TT.BEST_PATH}')
print()

# ── Launch training ───────────────────────────────────────
actor, critic = TT.train(
    n_rounds        = N_ROUNDS,
    games_per_round = GAMES_PER_ROUND,
    actor_path      = str(WORK / 'imitation' / 'checkpoints' / 'best_model.pt'),
    critic_path     = str(WORK / 'rl' / 'last.pt'),
    device          = DEVICE,
)

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Quick sanity evaluation (5 games vs self)
# ─────────────────────────────────────────────────────────
import random
from rl.transformer_trainer import play_game_collect, ValueNet

if actor is None:
    print('Training returned None — skipping eval.')
else:
    wins = losses = draws = 0
    # Use main.py as the 3 enemies for a quick check
    from rl.self_play_trainer import load_agent
    main_fn, _ = load_agent(str(WORK / 'main.py'))
    fns = [None, main_fn, main_fn, main_fn]  # player 0 = transformer

    print('Quick eval (5 games vs base main.py)...')
    for g in range(5):
        buf, outcome = play_game_collect(actor, critic, fns,
                                         seed=9000 + g, device=DEVICE)
        if   outcome == 'WIN':  wins   += 1
        elif outcome == 'LOSS': losses += 1
        else:                   draws  += 1
        print(f'  Game {g+1}: {outcome}  (W:{wins} L:{losses} D:{draws})')

    print(f'\nWin rate: {wins}/5 = {wins/5*100:.0f}%')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — List output files (download from Output tab)
# ─────────────────────────────────────────────────────────
import os

print('Output checkpoints:')
for pt in sorted((WORK / 'rl').glob('transformer*.pt')):
    size_mb = pt.stat().st_size / 1e6
    print(f'  {pt.name}  ({size_mb:.1f} MB)')

print()
print('To use in submission:')
print('  1. Download transformer_best.pt')
print('  2. Place in imitation/checkpoints/best_model.pt')
print('  3. Your agent_nn.py will pick it up automatically')

# ── Flush & close log ─────────────────────────────────────
try:
    import time as _t
    print(f'\n# Log ended : {_t.strftime("%Y-%m-%d %H:%M:%S UTC", _t.gmtime())}')
    _tee.flush()
    _log_sz = LOG_PATH.stat().st_size
    _tee.close()                       # restores sys.stdout to original
    print(f'\nLog saved   : {LOG_PATH}  ({_log_sz / 1024:.1f} KB)')
    print('Download ppo_training.log from the Output tab.')
except Exception as _log_err:
    print(f'Log close error: {_log_err}')